In [ ]:
^^# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Séance:2

Nom: OBOSSOU Richard

Porteur: P2

Etapes assignées:
2.2:la barre de bruit
2.6: les comparaisons



In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

FICHIERS = ["train.csv", "a_decider.csv", "HomeCredit_columns_description.csv"]

def trouver_dossier():
    for racine in [Path("/kaggle/input"), Path(".")]:
        if not racine.exists():
            continue
        for d in [racine, *racine.rglob("*")]:
            if d.is_dir() and all((d / f).is_file() for f in FICHIERS):
                return d
    raise FileNotFoundError(
        "Fichiers introuvables. Dans Kaggle : Add Input > Datasets > Your Datasets, puis relance. "
        "Fichiers vus dans /kaggle/input : "
        + str([str(p) for p in Path("/kaggle/input").rglob("*") if p.is_file()][:10])
    )

D = trouver_dossier()
print("dossier de données :", D)

tr = pd.read_csv(D / "train.csv")
ad = pd.read_csv(D / "a_decider.csv")
dico = pd.read_csv(D / "HomeCredit_columns_description.csv", encoding="latin-1")
dico = dico[dico.Table.str.startswith("application")]
print(tr.shape, ad.shape, len(dico))  # attendu : (246008, 122) (61503, 121) 122

dossier de données : /kaggle/input/datasets/richardobossou/home-credit1/Sujet_03_Home_Credit
(246008, 122) (61503, 121) 122


In [10]:
# Comparaison selon le ratio de coût (règle « EXT moyenne ») — cellule autonome
K = 30
y = tr.TARGET.to_numpy()
x = tr[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1).to_numpy(float)   # refuser si x est bas

def cout_ligne(r, yy, ratio):        # ratio si défaut accordé, 1 si bon client refusé
    return ratio * ((~r) & (yy == 1)) + 1.0 * (r & (yy == 0))

rng = np.random.RandomState(1)
lignes = []
for ratio in (2, 3, 5, 10):
    cr, ca, pr, rap = [], [], [], []
    for _ in range(K):
        idx = rng.permutation(len(y)); a, b = idx[: len(y)//2], idx[len(y)//2:]
        grille = np.nanquantile(x[a], np.arange(.02, .99, .02))
        t = grille[int(np.argmin([cout_ligne(x[a] < g, y[a], ratio).mean() for g in grille]))]
        r = x[b] < t                  # une valeur manquante n'est jamais refusée
        cr.append(cout_ligne(r, y[b], ratio).mean())
        ca.append(cout_ligne(np.zeros(len(b), bool), y[b], ratio).mean())
        pr.append(r.mean()); rap.append(r[y[b] == 1].mean())
    d = np.array(cr) - np.array(ca)
    lignes.append((ratio, np.mean(ca), np.mean(cr), np.std(cr), np.percentile(d, 2.5), np.percentile(d, 97.5),
                   np.mean(pr), np.mean(rap)))
sens_ratio = pd.DataFrame(lignes, columns=["ratio", "tout accorder", "règle EXT moyenne", "bruit",
                                           "écart IC95 bas", "écart IC95 haut", "part refusée", "défauts attrapés"])
print(sens_ratio.round(3).to_string(index=False))

 ratio  tout accorder  règle EXT moyenne  bruit  écart IC95 bas  écart IC95 haut  part refusée  défauts attrapés
     2          0.161              0.163  0.001           0.000            0.002         0.020             0.078
     3          0.243              0.238  0.002          -0.006           -0.004         0.021             0.079
     5          0.404              0.366  0.003          -0.041           -0.034         0.119             0.323
    10          0.807              0.583  0.003          -0.233           -0.215         0.288             0.577
